# FIT4112 – Lab 3 · Notebook 04 (TỰ CHỌN) – Rule-based Customer Clustering

**Mở rộng từ** [shop_cluster](https://github.com/TrangLe1912/shop_cluster). **Không thuộc yêu cầu để đạt 100 điểm Lab 3.**

**Câu hỏi nghiên cứu nhỏ:** Nếu dùng thêm hành vi mua các **cặp sản phẩm có luật kết hợp mạnh**, các cụm khách hàng có thay đổi đáng kể so với RFM không?

Quy trình:
1. Đọc giao dịch và kết quả RFM từ Lab 3 (không cần Papermill, không cần chạy lại Lab 1/2).
2. Khai phá luật kết hợp ở cấp **hóa đơn** bằng Apriori; chọn tối đa 8 **cặp sản phẩm độc nhất**.
3. Mỗi khách hàng có một đặc trưng 0/1 cho từng cặp: **có ít nhất một hóa đơn chứa cả hai sản phẩm**.
4. So sánh K-Means **RFM** với **RFM + các đặc trưng cặp sản phẩm** cùng một `k`.
5. Phân tích thay đổi nhãn cụm (ARI), kích thước cụm và top sản phẩm; không coi silhouette từ hai không gian khác nhau là phép so sánh chất lượng trực tiếp.

**Khác shop_cluster cũ:** không dùng 200 luật, không nhân thẳng giá trị lift lên feature, không đếm hai luật A→B và B→A thành hai đặc trưng giống nhau.

## 1. Đọc dữ liệu và cấu hình K-Means từ Notebook 02

Notebook 02 cần chạy trước để xuất `customer_segments.csv` và `selected_model.json`. Chúng ta dùng lại **đúng k và chế độ biến đổi RFM của sinh viên**, thay vì ngầm định một cấu hình khác.

In [ ]:
from pathlib import Path
from itertools import combinations
from collections import defaultdict
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mlxtend.frequent_patterns import apriori, association_rules
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

def find_root():
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / "data/raw/retail_transactions_demo.csv").exists():
            return folder
    raise FileNotFoundError("Không tìm thấy data/raw/retail_transactions_demo.csv")

ROOT = find_root()
OUT = ROOT / "data/processed"
seg_path = OUT / "customer_segments.csv"
config_path = OUT / "selected_model.json"
if not seg_path.exists() or not config_path.exists():
    raise FileNotFoundError("Hãy chạy Notebook 01 và 02 trước; cần customer_segments.csv và selected_model.json")

segments = pd.read_csv(seg_path, dtype={"CustomerID": "string"}).sort_values("CustomerID").reset_index(drop=True)
config = json.loads(config_path.read_text(encoding="utf-8"))
K = int(config["k"])
MODE = config["feature_mode"]
assert MODE in {"log1p+scaler", "scaler-only"}
assert 2 <= K < len(segments)
print("Cấu hình baseline:", config)
display(segments.head())

## 2. Chuẩn bị basket theo hóa đơn

Chỉ giữ giao dịch hợp lệ, giống Notebook 01; nhóm các mặt hàng theo `InvoiceNo`. Hai sản phẩm mua ở **hai hóa đơn khác nhau** không được tính là cùng giỏ.

In [ ]:
raw = pd.read_csv(
    ROOT / "data/raw/retail_transactions_demo.csv",
    dtype={"InvoiceNo": "string", "CustomerID": "string", "StockCode": "string"},
)
raw["InvoiceDate"] = pd.to_datetime(raw["InvoiceDate"], errors="coerce")
raw["Quantity"] = pd.to_numeric(raw["Quantity"], errors="coerce")
raw["UnitPrice"] = pd.to_numeric(raw["UnitPrice"], errors="coerce")
raw["CustomerID"] = raw["CustomerID"].str.strip()
valid = (
    raw["InvoiceNo"].notna()
    & ~raw["InvoiceNo"].str.startswith("C", na=False)
    & raw["CustomerID"].notna()
    & raw["CustomerID"].ne("")
    & raw["Description"].notna()
    & raw["Quantity"].gt(0)
    & raw["UnitPrice"].gt(0)
    & raw["Country"].eq("United Kingdom")
    & raw["InvoiceDate"].notna()
    & raw["InvoiceDate"].lt(pd.Timestamp("2026-07-01"))
)
clean = raw.loc[valid].copy()
clean["Description"] = clean["Description"].str.strip()
clean = clean[clean["CustomerID"].isin(segments["CustomerID"])]
assert clean["InvoiceNo"].groupby(clean["InvoiceNo"]).size().ge(1).all()
assert clean.groupby("InvoiceNo")["CustomerID"].nunique().eq(1).all(), "Một hóa đơn phải có một CustomerID."
basket = pd.crosstab(clean["InvoiceNo"], clean["Description"]).gt(0)
print("Basket:", basket.shape, "| Khách hàng:", clean["CustomerID"].nunique())
display(basket.head().astype(int))

## 3. Chọn tối đa 8 cặp từ luật kết hợp

**Tham số thử nghiệm:**

- `MIN_SUPPORT` = 0.06 – cặp hàng cần xuất hiện trong ít nhất 6% hóa đơn.
- `MIN_CONFIDENCE` = 0.35.
- `MAX_PAIRS` = 8.

Để đơn giản, chỉ xét luật **1 sản phẩm → 1 sản phẩm**. Chỉ giữ **một cặp vô hướng** cho hai luật đối xứng A→B và B→A. Phần luật vẫn có antecedents/consequents và các chỉ số để sinh viên đọc hiểu.

Đây là thí nghiệm mô tả trên cùng dữ liệu (in-sample), **không** phải mô hình kiểm định hiệu quả dự đoán trên khách hàng tương lai.

In [ ]:
MIN_SUPPORT = 0.06
MIN_CONFIDENCE = 0.35
MAX_PAIRS = 8

itemsets = apriori(basket, min_support=MIN_SUPPORT, use_colnames=True, max_len=2)
if itemsets.empty or not itemsets["itemsets"].map(len).ge(2).any():
    raise ValueError("Chưa tìm được cặp sản phẩm. Hãy giảm MIN_SUPPORT có giải thích.")
rules = association_rules(itemsets, metric="confidence", min_threshold=MIN_CONFIDENCE)
rules = rules.loc[
    rules["antecedents"].map(len).eq(1)
    & rules["consequents"].map(len).eq(1)
    & rules["lift"].gt(1)
].copy()
rules["pair"] = rules.apply(
    lambda r: tuple(sorted(set(r["antecedents"]) | set(r["consequents"]))), axis=1
)
# Tránh nhân đôi feature khi hai luật A→B và B→A dùng cùng một cặp sản phẩm
picked = (
    rules.sort_values(["lift", "support"], ascending=[False, False])
    .drop_duplicates("pair")
    .head(MAX_PAIRS)
    .reset_index(drop=True)
)
if len(picked) < 2:
    raise ValueError("Có quá ít cặp để minh họa; hãy điều chỉnh ngưỡng một cách có căn cứ.")
picked["ProductA"] = picked["pair"].str[0]
picked["ProductB"] = picked["pair"].str[1]
print("Số cặp sản phẩm được chọn:", len(picked))
display(picked[["ProductA", "ProductB", "support", "confidence", "lift"]].round(3))
picked[["ProductA", "ProductB", "support", "confidence", "lift"]].to_csv(
    OUT / "optional_selected_pairs.csv", index=False
)

## 4. Tạo đặc trưng Rule2Segment dễ giải thích

**Ý nghĩa đặc trưng:** `pair_j = 1` khi khách hàng **có ít nhất một hóa đơn mua cùng** hai sản phẩm của cặp j; ngược lại là 0.

Cách này chặt chẽ hơn kiểm tra khách đã từng mua A rồi từng mua B ở hai thời điểm khác nhau. Không dùng trọng số lift trực tiếp vì sẽ dễ làm mất cân bằng khoảng cách với RFM.

**Kiểm tra:** mọi CustomerID được giữ đúng thứ tự tương ứng với file phân cụm baseline.

In [ ]:
invoice_customers = clean[["InvoiceNo", "CustomerID"]].drop_duplicates().set_index("InvoiceNo")["CustomerID"]
pair_matrix = pd.DataFrame(0.0, index=segments["CustomerID"].astype(str), columns=[
    f"{a} + {b}" for a, b in picked["pair"]
])
for pair, col in zip(picked["pair"], pair_matrix.columns):
    a, b = pair
    cooccur = basket.index[basket[a] & basket[b]]
    customers = invoice_customers.loc[cooccur].astype(str).unique()
    pair_matrix.loc[pair_matrix.index.isin(customers), col] = 1.0
assert pair_matrix.shape == (len(segments), len(picked))
assert set(np.unique(pair_matrix.to_numpy())).issubset({0.0, 1.0})
print("Ma trận Customer × Pair:", pair_matrix.shape)
display(pair_matrix.head().astype(int))

## 5. Phân cụm với RFM và RFM + Rule Features

Cả hai mô hình dùng **cùng k** và cùng cách biến đổi RFM. Để không để block rule có nhiều cột tự động áp đảo 3 đặc trưng RFM:

- RFM: StandardScaler, chia cho `sqrt(3)`.
- Rule: StandardScaler, chia cho `sqrt(n_pairs)`.
- Nhân block rule thêm hệ số `RULE_WEIGHT = 0.5` (giá trị minh họa, không tối ưu sẵn).

Đây là quy tắc cân bằng theo khối để thử nghiệm, **không** chứng minh đó là cách chọn trọng số tối ưu.

In [ ]:
features = ["Recency", "Frequency", "Monetary"]
rfm = segments[features].astype(float)
rfm_input = np.log1p(rfm) if MODE == "log1p+scaler" else rfm
x_rfm = StandardScaler().fit_transform(rfm_input) / np.sqrt(len(features))
x_pairs = StandardScaler().fit_transform(pair_matrix) / np.sqrt(pair_matrix.shape[1])
RULE_WEIGHT = 0.5  # TODO: thử 0.25 hoặc 1.0 nếu muốn
x_aug = np.hstack([x_rfm, RULE_WEIGHT * x_pairs])

clusterer = lambda x: KMeans(
    n_clusters=K, random_state=int(config.get("random_state", 42)),
    n_init=int(config.get("n_init", 10))
).fit_predict(x)
baseline_labels = clusterer(x_rfm)
augmented_labels = clusterer(x_aug)

# Kiểm tra baseline đúng như Notebook 02 (phép chia đều toàn bộ RFM không đổi nghiệm K-Means).
reference_labels = segments["Cluster"].to_numpy()
baseline_match = adjusted_rand_score(reference_labels, baseline_labels)
print("ARI (baseline Notebook 02 và baseline tính lại):", round(baseline_match, 3))
assert baseline_match > 0.999, "Cấu hình baseline không trùng Notebook 02; hãy chạy lại Notebook 02."

ari = adjusted_rand_score(baseline_labels, augmented_labels)
result = segments[["CustomerID"] + features].copy()
result["RFMCluster"] = baseline_labels
result["AugmentedCluster"] = augmented_labels
print("ARI giữa hai phương án phân cụm:", round(ari, 3))
print("ARI=1: hai phép chia cụm tương đương (không phụ thuộc số nhãn 0/1/2).")
display(pd.crosstab(result["RFMCluster"], result["AugmentedCluster"]))
result.to_csv(OUT / "optional_rule_based_clusters.csv", index=False)

## 6. Diễn giải kết quả và giới hạn

**Quan trọng:** Silhouette của hai phương án được tính trên **hai không gian đặc trưng khác nhau**. Có thể ghi nhận, nhưng **không thể so sánh trực tiếp** để kết luận bổ sung rule là tốt hơn.

Thay vì vậy, hãy trình bày:
- Tỷ lệ khách hàng đổi nhóm theo bảng chéo (lưu ý nhãn cụm là tùy ý).
- ARI (bất biến với hoán vị nhãn).
- RFM trung vị và sở thích sản phẩm của các nhóm mới.
- Khả năng giải thích và giá trị sử dụng thực tế của phân khúc.

In [ ]:
summary = result.groupby("AugmentedCluster").agg(
    Customers=("CustomerID", "size"),
    Recency_median=("Recency", "median"),
    Frequency_median=("Frequency", "median"),
    Monetary_median=("Monetary", "median"),
).reset_index()
display(summary.round(2))
print("Silhouette RFM (không gian RFM):", round(silhouette_score(x_rfm, baseline_labels), 3))
print("Silhouette RFM+Rule (không gian mở rộng):", round(silhouette_score(x_aug, augmented_labels), 3))
print("Không dùng hai silhouette trên hai không gian khác nhau làm căn cứ kết luận mô hình nào tốt hơn.")
summary.to_csv(OUT / "optional_augmented_profiles.csv", index=False)

## TODO – Bài tập nâng cao tự chọn

1. Chọn 3 cặp sản phẩm trong bảng luật và diễn giải support/confidence/lift. Cặp nào có giá trị kinh doanh?
2. Có bao nhiêu đặc trưng rule? Vì sao dùng **cặp sản phẩm duy nhất** thay cho 200 luật có hướng như `shop_cluster`?
3. So sánh nhãn phân cụm RFM và RFM+Rule qua **ARI**, bảng chéo, profile. Nếu ARI nhỏ, có nhất thiết là mô hình tốt hơn không?
4. Giải thích vì sao chuẩn hóa từng **khối** đặc trưng và tại sao cần `RULE_WEIGHT`.
5. Nêu **ít nhất hai hạn chế**: dữ liệu demo giả lập; luật khai phá và phân cụm trên cùng tập giao dịch; số k cố định; không có thử nghiệm kinh doanh ngoài thực tế.

**Phần này hoàn toàn tự chọn** cho HTTT hoặc PTPM; không làm tăng yêu cầu bắt buộc của rubric Lab 3.